In [ ]:
import numpy as np 
import pandas as pd 
import os
import pydicom
from glob import glob
from collections import defaultdict

DATA_PATH = '/kaggle/input/competitions/rsna-knee-abnormality-detection'

class DICOMExtractor :
    def __init__(self, data_path) :
        self.data_path = data_path

    def _getStudyInstanceUID(self, file = 'train.csv') -> list[str] :
        df = pd.read_csv(os.path.join(self.data_path, file))
        return df['StudyInstanceUID'].to_list()

    def _getSeriesInstanceUID(self, file = 'train_series.csv') -> dict:
        df = pd.read_csv(os.path.join(self.data_path, file))
        seriesInstanceUID = defaultdict(list)
        for study, series in zip(df['StudyInstanceUID'], df['SeriesInstanceUID']) :
            seriesInstanceUID[study].append(series)

        return seriesInstanceUID

    
    def getDICOM(self, metadata_only = False, file = 'train_series'):
        dicomInstances = {}
        for study, series in self._getSeriesInstanceUID().items():
            for ser in series:
                series_dir = os.path.join(self.data_path, file, study, ser)
                paths = sorted(glob(os.path.join(series_dir, "*.dcm")))
                
                if not paths:
                    continue
    
                datasets = []
                for p in paths:
                    try:
                        datasets.append(pydicom.dcmread(p, stop_before_pixels=metadata_only))
                    except Exception as e:
                        print(f"skipping {p}: {type(e).__name__}: {e}")
    
                if not datasets:
                    continue
    
                iop = np.array(datasets[0].ImageOrientationPatient, float)
                normal = np.cross(iop[:3], iop[3:])
                datasets.sort(key=lambda d: float(np.dot(np.array(d.ImagePositionPatient, float), normal)))
                yield (study, ser), datasets
        


    
if __name__ == '__main__' :
    d = DICOMExtractor(DATA_PATH)
    print(d)
    print(next(d.getDICOM()))
    

In [6]:
# -*- coding: utf-8 -*-
"""Rule-based label extraction from multilingual radiology reports.

Turns a free-text knee MRI report, in any of ~12 languages, into 12 soft
labels in [0, 1] suitable for training a vision model.

    labeler = ClinicalNoteLabeler()
    labeler.to_soft_labels("No ACL tear. Medial meniscus posterior horn tear.")
    # {'ACL': 0.02, 'medial_meniscus': 0.95, ...}

Stdlib only. No model, no network, no GPU.

DESIGN
------
Six small pieces rather than one big class, so each can be tested and swapped
independently:

    Certainty          the six-level ordinal scale and its mapping to numbers
    Mention            one occurrence of one finding in one text unit
    LabelResult        the final per-label verdict, with provenance
    Vocabulary         all language-specific data, isolated from all logic
    TextNormalizer     script detection, accent folding, abbreviation expansion
    NegationDetector   scope windows and polarity
    ClinicalNoteLabeler  orchestrates the above

The split that matters most is Vocabulary vs the detectors. Adding a language
should mean adding data, never touching logic. If you find yourself editing
NegationDetector to support Polish, something is wrong with the boundary.
"""
from __future__ import annotations

import re
import unicodedata
from dataclasses import dataclass, field
from typing import Iterable, Iterator, Sequence

__all__ = [
    "LABELS", "Certainty", "Mention", "LabelResult",
    "Vocabulary", "TextNormalizer", "NegationDetector", "ClinicalNoteLabeler",
]

# The 12 findings. Order is the submission column order; nothing else in this
# module hardcodes a label list.
LABELS: list[str] = [
    "ACL", "MCL", "medial_meniscus", "lateral_meniscus",
    "medial_OA", "lateral_OA", "patellofemoral_OA",
    "effusion", "synovitis", "bakers_cyst", "bone_contusion", "fracture",
]


# =============================================================================
# 1. Certainty
# =============================================================================
class Certainty:
    """The ordinal scale a report can express about a finding.

    Six levels rather than a boolean, because reports hedge constantly and
    collapsing "definite tear" and "tear cannot be excluded" to the same 1
    throws away information the model can use.

    Categorical rather than a raw float because the mapping to numbers is a
    tunable you want in one place, calibrated against a gold set, not scattered
    through the matching code.
    """

    DEFINITE = "definite"
    PROBABLE = "probable"
    POSSIBLE = "possible"
    UNLIKELY = "unlikely"
    NEGATED = "negated"
    NOT_MENTIONED = "not_mentioned"
    UNSUPPORTED = "script_unsupported"     # see ClinicalNoteLabeler.extract

    # Strength ordering. Used to pick a winner when one report mentions the
    # same finding more than once.
    RANK = {
        DEFINITE: 5, PROBABLE: 4, POSSIBLE: 3,
        UNLIKELY: 2, NEGATED: 1, NOT_MENTIONED: 0, UNSUPPORTED: 0,
    }

    # Default categorical -> probability map. Override per project.
    DEFAULT_VALUES = {
        DEFINITE: 0.95, PROBABLE: 0.80, POSSIBLE: 0.50,
        UNLIKELY: 0.20, NEGATED: 0.02, NOT_MENTIONED: 0.02, UNSUPPORTED: 0.02,
    }

    @classmethod
    def stronger(cls, a: str, b: str) -> str:
        return a if cls.RANK[a] >= cls.RANK[b] else b


# =============================================================================
# 2/3. Value objects
# =============================================================================
@dataclass
class Mention:
    """One occurrence of one finding inside one text unit.

    Carries the unit it was found in so downstream code can show evidence
    without re-parsing, and so negation can be scoped without passing the whole
    document around.
    """

    label: str
    matched_text: str
    span: tuple[int, int]
    unit: str
    certainty: str = Certainty.DEFINITE
    negation_source: str = ""       # "", "pre", "post"

    def evidence(self, max_chars: int = 200) -> str:
        return self.unit.strip()[:max_chars]


@dataclass
class LabelResult:
    """Final verdict for one label on one report."""

    label: str
    certainty: str
    value: float
    evidence: str = ""
    needs_review: bool = False

    @property
    def is_positive(self) -> bool:
        return self.value > 0.5


# =============================================================================
# 4. Vocabulary — all language data, no logic
# =============================================================================
@dataclass
class Vocabulary:
    """Language-specific data for matching.

    TERM FORMAT: space-separated STEMS, not dictionary forms.
    "медиальн мениск" not "медиальный мениск". The compiler below turns each
    stem into `stem\\w*`, so every inflected form matches. This is essential for
    Russian and Greek, where an adjective-noun pair inflects on BOTH words and
    a literal substring search finds nothing.

    ACCURACY WARNING: the non-English entries below are a starting point, not
    validated terminology. Check them against your actual corpus before
    trusting them. A wrong stem produces no match, and no match looks exactly
    like a negative finding.
    """

    findings: dict[str, list[str]] = field(default_factory=dict)
    pre_negation: list[str] = field(default_factory=list)
    post_negation: list[str] = field(default_factory=list)
    hedges: dict[str, list[str]] = field(default_factory=dict)
    abbreviations: dict[str, str] = field(default_factory=dict)
    supported_scripts: set[str] = field(default_factory=lambda: {"latin", "greek", "cyrillic"})

    _compiled: dict[str, list[re.Pattern]] = field(default_factory=dict, repr=False)

    # -- compilation ------------------------------------------------------
    @staticmethod
    def compile_term(term: str) -> re.Pattern:
        r"""Turn "медиальн мениск" into `медиальн\w*[\s\-]*мениск\w*`.

        Each stem gets a trailing \w* so any case/number ending matches, and
        the separator tolerates a space or hyphen. Latin terms are unaffected:
        "medial meniscus" still matches itself, and now also "mediale
        meniscus" for free.
        """
        stems = [re.escape(s) for s in term.split()]
        return re.compile(r"\w*[\s\-]*".join(stems) + r"\w*", re.I | re.U)

    def compiled(self, label: str) -> list[re.Pattern]:
        """Lazily compile and cache patterns for one label."""
        if label not in self._compiled:
            self._compiled[label] = [self.compile_term(t) for t in self.findings.get(label, [])]
        return self._compiled[label]

    def add_language(self, findings: dict[str, list[str]],
                     pre: Sequence[str] = (), post: Sequence[str] = (),
                     hedges: dict[str, list[str]] | None = None) -> "Vocabulary":
        """Merge another language in. Returns self so calls chain.

        Scripts cannot collide -- a Cyrillic stem will never match Latin text --
        so everything lives in one merged pool and there is no routing by
        language at match time. That also means a mixed-language report works
        without any special handling.
        """
        for lab, terms in findings.items():
            self.findings.setdefault(lab, []).extend(terms)
        self.pre_negation.extend(pre)
        self.post_negation.extend(post)
        for bucket, cues in (hedges or {}).items():
            self.hedges.setdefault(bucket, []).extend(cues)
        self._compiled.clear()
        return self


def build_default_vocabulary() -> Vocabulary:
    """English/German/French/Spanish/Dutch + Greek + Russian."""
    v = Vocabulary(
        findings={
            "ACL": ["anterior cruciate"],
            "MCL": ["medial collateral"],
            "medial_meniscus": ["medial meniscus"],
            "lateral_meniscus": ["lateral meniscus"],
            "medial_OA": ["medial osteoarthritis", "medial compartment osteoarthritis",
                          "medial chondral loss"],
            "lateral_OA": ["lateral osteoarthritis", "lateral compartment osteoarthritis",
                           "lateral chondral loss"],
            "patellofemoral_OA": ["patellofemoral", "retropatellar", "chondromalacia patell"],
            "effusion": ["effusion", "joint fluid"],
            "synovitis": ["synovitis", "synovial thickening"],
            "bakers_cyst": ["baker cyst", "bakers cyst", "popliteal cyst"],
            "bone_contusion": ["bone marrow edema", "bone marrow oedema", "bone contusion",
                               "bone bruise"],
            "fracture": ["fracture", "avulsion"],
        },
        pre_negation=[r"\bno\b", r"\bnot\b", r"\bwithout\b", r"\bnegative for\b",
                      r"\babsence of\b", r"\bfree of\b", r"\bruled out\b", r"\bexcluded\b"],
        post_negation=[r"\bintact\b", r"\bnormal\b", r"\bunremarkable\b", r"\bpreserved\b",
                       r"\bwithin normal limits\b", r"\bwnl\b"],
        hedges={
            "probable": [r"likely", r"probable", r"consistent with", r"suggestive of"],
            "possible": [r"possible", r"cannot be (excluded|ruled out)", r"suspicion",
                         r"query", r"may represent", r"\bversus\b", r"\bvs\b"],
            "unlikely": [r"unlikely", r"doubtful"],
        },
        abbreviations={
            r"\bACL\b": "anterior cruciate ligament", r"\bVKB\b": "anterior cruciate ligament",
            r"\bLCA\b": "anterior cruciate ligament", r"\bMCL\b": "medial collateral ligament",
            r"\bMM\b": "medial meniscus", r"\bLM\b": "lateral meniscus",
            r"\bPF\b": "patellofemoral", r"\bOA\b": "osteoarthritis",
            r"\beff\b": "effusion", r"\bBME\b": "bone marrow edema",
            r"\bfx\b": "fracture", r"\bsyn\b": "synovitis",
        },
    )

    v.add_language(
        {   # German
            "ACL": ["vorder kreuzband", "kreuzband vorder"],
            "MCL": ["innenband", "mediale kollateralband"],
            "medial_meniscus": ["innenmeniskus"],
            "lateral_meniscus": ["aussenmeniskus", "außenmeniskus"],
            "medial_OA": ["mediale gonarthrose", "mediale arthrose"],
            "lateral_OA": ["laterale gonarthrose", "laterale arthrose"],
            "effusion": ["erguss", "gelenkerguss"],
            "bakers_cyst": ["bakerzyste"],
            "bone_contusion": ["knochenmarkodem", "knochenmarködem"],
            "fracture": ["fraktur"],
        },
        pre=[r"\bkein\b", r"\bkeine\b", r"\bohne\b"],
        post=[r"\bunauffallig\b", r"\bintakt\b", r"\bregelrecht\b"],
        hedges={"possible": [r"verdacht", r"moglich"], "probable": [r"wahrscheinlich"]},
    ).add_language(
        {   # French
            "ACL": ["ligament croise anterieur"],
            "medial_meniscus": ["menisque medial", "menisque interne"],
            "lateral_meniscus": ["menisque lateral", "menisque externe"],
            "effusion": ["epanchement"],
            "bakers_cyst": ["kyste de baker"],
            "bone_contusion": ["oedeme osseux"],
        },
        pre=[r"\bpas de\b", r"\bsans\b", r"\baucun\b"],
        post=[r"\bsans particularite\b", r"\bnormale?\b"],
        hedges={"possible": [r"possible", r"ne peut etre exclu", r"suspicion"],
                "probable": [r"compatible avec", r"en faveur de"]},
    ).add_language(
        {   # Spanish
            "ACL": ["ligamento cruzado anterior"],
            "medial_meniscus": ["menisco medial", "menisco interno"],
            "lateral_meniscus": ["menisco lateral", "menisco externo"],
            "effusion": ["derrame"],
            "bakers_cyst": ["quiste de baker"],
            "bone_contusion": ["edema oseo"],
            "fracture": ["fisura"],
        },
        pre=[r"\bno hay\b", r"\bsin\b", r"\bausencia\b"],
        post=[r"\bintacto\b", r"\bnormales?\b"],
        hedges={"possible": [r"posible", r"no se puede excluir", r"sospecha"],
                "probable": [r"compatible con", r"sugestivo de"]},
    ).add_language(
        {   # Dutch
            "ACL": ["voorste kruisband"],
            "medial_meniscus": ["mediale meniscus", "binnenmeniscus"],
            "lateral_meniscus": ["laterale meniscus", "buitenmeniscus"],
        },
        pre=[r"\bgeen\b", r"\bzonder\b"],
        post=[r"\bintact\b", r"\bnormaal\b"],
    ).add_language(
        {   # Russian (Cyrillic) -- stems
            "ACL": ["передн крестообразн", "пкс"],
            "MCL": ["внутренн боков", "медиальн боков"],
            "medial_meniscus": ["медиальн мениск", "внутренн мениск"],
            "lateral_meniscus": ["латеральн мениск", "наружн мениск"],
            "medial_OA": ["медиальн артроз", "внутренн артроз"],
            "lateral_OA": ["латеральн артроз", "наружн артроз"],
            "patellofemoral_OA": ["пателлофеморальн", "ретропателляр"],
            "effusion": ["выпот", "жидкост в полост"],
            "synovitis": ["синовит"],
            "bakers_cyst": ["киста бейкер", "подколенн киста"],
            "bone_contusion": ["отек костн мозг", "трабекулярн отек"],
            "fracture": ["перелом", "трещин"],
        },
        pre=[r"\bне\b", r"\bнет\b", r"\bбез\b"],
        # These are POST-posed in Russian: "перелом не выявлен" puts the
        # negation after the finding, unlike English "no fracture".
        post=[r"интактн", r"сохранн", r"не изменен", r"в норме", r"нормальн",
              r"не выявлен", r"не определя", r"не отмеча", r"не обнаружен", r"отсутств"],
        hedges={"possible": [r"возможн", r"подозрени", r"не исключ"],
                "probable": [r"вероятн", r"соответству"]},
    ).add_language(
        {   # Greek -- stems, written unaccented (normaliser strips tonos)
            "ACL": ["προσθι χιαστ", "χιαστου συνδεσμ"],
            "MCL": ["εσω πλαγι συνδεσμ"],
            "medial_meniscus": ["εσω μηνισκ", "εσωτερικ μηνισκ"],
            "lateral_meniscus": ["εξω μηνισκ", "εξωτερικ μηνισκ"],
            "medial_OA": ["εσω οστεοαρθρι"],
            "lateral_OA": ["εξω οστεοαρθρι"],
            "patellofemoral_OA": ["επιγονατιδομηριαι", "οπισθοεπιγονατιδ"],
            "effusion": ["αρθρικ συλλογ", "ενδαρθρικ υγρ"],
            "synovitis": ["υμενιτιδ"],
            "bakers_cyst": ["κυστ baker", "ιγνυακ κυστ"],
            "bone_contusion": ["οιδημα μυελ", "οστικ οιδημα"],
            "fracture": ["καταγμα", "ρωγμ"],
        },
        pre=[r"\bδεν\b", r"\bχωρις\b", r"απουσι", r"ουδεμι"],
        post=[r"ακεραι", r"φυσιολογικ", r"ανευ ευρηματ"],
        hedges={"possible": [r"πιθανον", r"δεν αποκλειετ"], "probable": [r"συμβατ"]},
    )
    return v


# =============================================================================
# 5. TextNormalizer
# =============================================================================
class TextNormalizer:
    """Script detection, accent folding, abbreviation expansion, unit splitting."""

    SCRIPT_RANGES = {
        "greek": (0x0370, 0x03FF), "cyrillic": (0x0400, 0x04FF),
        "arabic": (0x0600, 0x06FF), "hebrew": (0x0590, 0x05FF),
        "devanagari": (0x0900, 0x097F), "han": (0x4E00, 0x9FFF),
        "kana": (0x3040, 0x30FF), "hangul": (0xAC00, 0xD7AF),
    }

    # A unit boundary. Negation must not cross one.
    UNIT_SPLIT = re.compile(r"[\n\r]+|(?<=[.;:])\s+")

    def __init__(self, abbreviations: dict[str, str] | None = None):
        self.abbreviations = abbreviations or {}

    def detect_script(self, text: str) -> str:
        """Dominant script by letter census.

        Dominance, not presence: reports routinely mix Cyrillic prose with
        Latin units and drug names, so an any() test would misclassify them.
        """
        counts = {k: 0 for k in self.SCRIPT_RANGES}
        counts["latin"] = 0
        for ch in text or "":
            if not ch.isalpha():
                continue
            o = ord(ch)
            if o < 0x0250:
                counts["latin"] += 1
                continue
            for name, (lo, hi) in self.SCRIPT_RANGES.items():
                if lo <= o <= hi:
                    counts[name] += 1
                    break
        return max(counts, key=counts.get) if any(counts.values()) else "unknown"

    def normalize(self, text: str) -> str:
        """Fold accents, normalise Greek sigma, expand abbreviations, tidy space.

        NFKD decomposition plus combining-mark stripping folds Latin diacritics
        (é -> e) AND Greek tonos (ά -> α), which is exactly what we want since
        the Greek vocabulary is written unaccented. For Cyrillic it folds
        ё -> е and й -> и, harmless because the vocabulary goes through the
        same function.
        """
        if not text:
            return ""
        t = unicodedata.normalize("NFKD", text)
        t = "".join(c for c in t if not unicodedata.combining(c))
        t = t.replace("\u03c2", "\u03c3")        # Greek final sigma -> sigma
        for pat, rep in self.abbreviations.items():
            t = re.sub(pat, rep, t, flags=re.I)
        return re.sub(r"[ \t]+", " ", t)

    def split_units(self, text: str) -> list[str]:
        """Sentences or lines. The unit is the negation scope."""
        return [u.strip() for u in self.UNIT_SPLIT.split(text) if u.strip()]


# =============================================================================
# 6. NegationDetector
# =============================================================================
class NegationDetector:
    """Decides whether a mention is asserted, denied, or hedged.

    THE CENTRAL PROBLEM. "No evidence of ACL tear" contains "ACL tear". Since
    most mentions of most findings in radiology reports are negative, a matcher
    that cannot tell assertion from denial is worse than predicting the base
    rate.

    Two directions, because languages put the cue on different sides:
        pre-posed   "no fracture"            (English, German, French, Spanish)
        post-posed  "the ACL is intact"      (English)
                    "перелом не выявлен"     (Russian -- fracture not detected)

    A single backwards search gets post-posed negation exactly backwards,
    labelling intact structures as torn.
    """

    # Clause boundaries within a unit. "No fracture, but ACL torn" must not
    # negate the ACL.
    CLAUSE_BREAK = re.compile(
        r"[,;:]|\bbut\b|\bhowever\b|\baber\b|\bjedoch\b|\bmais\b|\bpero\b|\bmaar\b|\bно\b|\bαλλα\b",
        re.I | re.U,
    )

    def __init__(self, vocab: Vocabulary, window: int = 80):
        self.vocab = vocab
        self.window = window

    def _scopes(self, unit: str, span: tuple[int, int]) -> tuple[str, str]:
        """Left and right context, truncated at the nearest clause break."""
        left = unit[max(0, span[0] - self.window): span[0]].lower()
        breaks = list(self.CLAUSE_BREAK.finditer(left))
        if breaks:
            left = left[breaks[-1].end():]

        right = unit[span[1]: span[1] + self.window].lower()
        brk = self.CLAUSE_BREAK.search(right)
        if brk:
            right = right[: brk.start()]
        return left, right

    def detect(self, mention: Mention) -> str:
        """Returns '' (not negated), 'pre', or 'post'."""
        left, right = self._scopes(mention.unit, mention.span)
        if any(re.search(c, left, re.U) for c in self.vocab.pre_negation):
            return "pre"
        if any(re.search(c, right, re.U) for c in self.vocab.post_negation):
            return "post"
        return ""

    def certainty(self, mention: Mention) -> str:
        """Hedge level for a non-negated mention."""
        left, right = self._scopes(mention.unit, mention.span)
        ctx = left + " " + right
        for bucket, cues in self.vocab.hedges.items():
            if any(re.search(c, ctx, re.U) for c in cues):
                return bucket
        return Certainty.DEFINITE


# =============================================================================
# 7. ClinicalNoteLabeler
# =============================================================================
class ClinicalNoteLabeler:
    """Orchestrates normalisation, matching, negation and scoring.

        labeler = ClinicalNoteLabeler()
        results = labeler.extract(report)          # dict[label] -> LabelResult
        soft    = labeler.to_soft_labels(report)   # dict[label] -> float
        df_rows = list(labeler.batch(pairs))       # for a training CSV
    """

    def __init__(self, vocabulary: Vocabulary | None = None,
                 certainty_values: dict[str, float] | None = None,
                 omission_priors: dict[str, float] | None = None,
                 labels: Sequence[str] = LABELS):
        """
        certainty_values  categorical -> probability. Tune on a gold set.
        omission_priors   per-label p(present | never mentioned). See below.
        """
        self.vocab = vocabulary or build_default_vocabulary()
        self.values = dict(Certainty.DEFAULT_VALUES)
        if certainty_values:
            self.values.update(certainty_values)
        self.omission_priors = omission_priors or {}
        self.labels = list(labels)
        self.normalizer = TextNormalizer(self.vocab.abbreviations)
        self.negation = NegationDetector(self.vocab)

    # -- matching ---------------------------------------------------------
    def find_mentions(self, unit: str) -> list[Mention]:
        """All findings mentioned in one text unit.

        First matching term per label wins and we move on -- the terms within a
        label are synonyms, so a second hit adds nothing.
        """
        out = []
        for label in self.labels:
            for pat in self.vocab.compiled(label):
                m = pat.search(unit)
                if m:
                    out.append(Mention(label, m.group(0), m.span(), unit))
                    break
        return out

    # -- main entry point -------------------------------------------------
    def extract(self, text: str) -> dict[str, LabelResult]:
        script = self.normalizer.detect_script(text)
        normalized = self.normalizer.normalize(text)

        mentions: list[Mention] = []
        for unit in self.normalizer.split_units(normalized):
            for m in self.find_mentions(unit):
                src = self.negation.detect(m)
                m.negation_source = src
                m.certainty = Certainty.NEGATED if src else self.negation.certainty(m)
                mentions.append(m)

        # One report can mention a finding several times ("possible medial
        # meniscus tear" in findings, "medial meniscus tear" in impression).
        # The strongest assertion wins: a report that both hedges and asserts
        # has resolved its own uncertainty by the time it asserts.
        best: dict[str, Mention] = {}
        for m in mentions:
            cur = best.get(m.label)
            if cur is None or Certainty.RANK[m.certainty] > Certainty.RANK[cur.certainty]:
                best[m.label] = m

        # COVERAGE GUARD.
        # Zero mentions across a whole report means one of two things: a
        # genuinely unremarkable study, or a language this vocabulary does not
        # cover. Those produce identical output -- all negative -- and must not
        # be conflated, because the second silently poisons the training set
        # with an entire language's worth of false negatives. Flagging it turns
        # a silent failure into a visible one.
        unsupported = not mentions and script not in self.vocab.supported_scripts

        results = {}
        for label in self.labels:
            m = best.get(label)
            if m is None:
                cert = Certainty.UNSUPPORTED if unsupported else Certainty.NOT_MENTIONED
                val = self.omission_priors.get(label, self.values[Certainty.NOT_MENTIONED])
                results[label] = LabelResult(label, cert, val, "", unsupported)
            else:
                results[label] = LabelResult(
                    label, m.certainty, self.values[m.certainty], m.evidence(), False
                )
        return results

    def to_soft_labels(self, text: str) -> dict[str, float]:
        return {k: v.value for k, v in self.extract(text).items()}

    # -- batch helpers ----------------------------------------------------
    def batch(self, reports: Iterable[tuple[str, str]]) -> Iterator[dict]:
        """Yield one flat row per report. Feed straight to pd.DataFrame.

        A generator so a corpus of any size streams rather than materialising.
        """
        for study_id, text in reports:
            res = self.extract(text)
            yield {"study_id": study_id,
                   **{lab: r.value for lab, r in res.items()}}

    def review_queue(self, reports: Iterable[tuple[str, str]]) -> list[dict]:
        """Reports the extractor could not handle, for a human to look at.

        With a hand-built multilingual vocabulary this is the most useful
        diagnostic in the module: it tells you which languages or templates you
        are silently failing on, ranked so the worst come first.
        """
        rows = []
        for study_id, text in reports:
            res = self.extract(text)
            flagged = [r.label for r in res.values() if r.needs_review]
            if flagged:
                rows.append({
                    "study_id": study_id,
                    "script": self.normalizer.detect_script(text),
                    "n_flagged": len(flagged),
                    "preview": (text or "")[:80],
                })
        return sorted(rows, key=lambda r: -r["n_flagged"])

    def coverage_report(self, reports: Iterable[tuple[str, str]]) -> dict:
        """Per-script hit rate. Run this before trusting any output.

        A script with a near-zero mention rate is a vocabulary gap, not a
        population of healthy knees.
        """
        stats: dict[str, dict] = {}
        for _, text in reports:
            script = self.normalizer.detect_script(text)
            s = stats.setdefault(script, {"n": 0, "with_mentions": 0})
            s["n"] += 1
            res = self.extract(text)
            if any(r.certainty not in (Certainty.NOT_MENTIONED, Certainty.UNSUPPORTED)
                   for r in res.values()):
                s["with_mentions"] += 1
        for s in stats.values():
            s["hit_rate"] = round(s["with_mentions"] / max(s["n"], 1), 3)
        return stats


In [8]:
labeler = ClinicalNoteLabeler()
labeler.to_soft_labels("No Medial meniscus. No ACL tear found")

{'ACL': 0.02,
 'MCL': 0.02,
 'medial_meniscus': 0.02,
 'lateral_meniscus': 0.02,
 'medial_OA': 0.02,
 'lateral_OA': 0.02,
 'patellofemoral_OA': 0.02,
 'effusion': 0.02,
 'synovitis': 0.02,
 'bakers_cyst': 0.02,
 'bone_contusion': 0.02,
 'fracture': 0.02}

In [ ]:
# -*- coding: utf-8 -*-
"""Join the pixel side and the text side into one dataset.

`DICOMExtractor` knows where the images are. `ClinicalNoteLabeler` knows what
the report says. Nothing so far knows that a study has both. This module is
that missing piece:

    builder = KneeDatasetBuilder(DATA_PATH)
    df = builder.build()                  # one row per series: meta + labels
    ds = builder.dataset()                # indexable; volumes read on demand
    vol, y, meta = ds[0]["volume"], ds[0]["labels"], ds[0]["meta"]

DESIGN
------
Three ideas carry the whole thing:

1.  THE ROW IS A SERIES, THE LABEL IS A STUDY.
    Reports are written per study; pixels live per series; a knee study is
    typically 4-8 sequences. So labels are broadcast down to every series of
    their study and the study id is kept on every row, which is what you group
    by when you split train/val. Splitting on series leaks a study's report
    across the split and flatters your validation score. `build()` also emits
    `n_series_in_study` so a per-study aggregation is one groupby away.

2.  METADATA IS EAGER, PIXELS ARE LAZY.
    One header per slice is cheap and gives you everything you filter on
    (plane, weighting, slice count, spacing). Pixels are gigabytes. So the
    scan reads headers only (`stop_before_pixels=True`) and stores the sorted
    file paths; `SeriesRecord.volume()` reads pixels when you actually ask.
    The table is therefore small enough to cache to disk and re-load in
    seconds, which is the difference between iterating on your dataset and
    waiting on it.

3.  A FAILED SERIES IS DATA, NOT AN EXCEPTION.
    Unreadable files, missing folders, absent geometry tags, ragged slice
    shapes: on a real DICOM corpus all of these happen, and a scan that dies
    on the first one is useless. Every failure is recorded on the row
    (`scan_error`, `missing_files`) and surfaced in `builder.errors`, so a bad
    series is something you can count and inspect rather than something that
    ends the run.
"""
from __future__ import annotations

import json
import os
import re
import warnings
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass, field
from glob import glob
from typing import Any, Callable, Iterable, Iterator, Sequence

import numpy as np
import pandas as pd
import pydicom

__all__ = [
    "SeriesRecord", "KneeDataset", "KneeDatasetBuilder",
    "build_knee_dataset", "sort_dicom_paths", "scan_series",
]


# =============================================================================
# 1. DICOM helpers  (free functions: usable on their own, easy to test)
# =============================================================================
def _get(ds, *names, default=None):
    """First present, non-empty attribute out of `names`.

    Vendors disagree on which tag carries a concept -- laterality is
    `ImageLaterality` on one scanner and `Laterality` on the next -- so every
    read goes through a list of candidates instead of one attribute access.
    """
    for n in names:
        v = getattr(ds, n, None)
        if v is not None and v != "":
            return v
    return default


def _num(v, default=None):
    """DICOM numbers arrive as DSfloat, IS, str, or a 1-element list."""
    if v is None:
        return default
    if isinstance(v, (list, tuple, pydicom.multival.MultiValue)):
        v = v[0] if len(v) else None
    try:
        return float(v)
    except (TypeError, ValueError):
        return default


def _age_years(value) -> float | None:
    """'034Y' -> 34.0, '018M' -> 1.5. Returns None on anything else."""
    if not value:
        return None
    m = re.fullmatch(r"(\d{1,3})\s*([DWMY])?", str(value).strip(), re.I)
    if not m:
        return _num(value)
    n = float(m.group(1))
    return n / {"D": 365.25, "W": 52.0, "M": 12.0, "Y": 1.0}[(m.group(2) or "Y").upper()]


def slice_normal(ds) -> np.ndarray | None:
    """Unit vector out of the image plane, from ImageOrientationPatient."""
    iop = _get(ds, "ImageOrientationPatient")
    if iop is None or len(iop) != 6:
        return None
    iop = np.asarray(iop, dtype=float)
    return np.cross(iop[:3], iop[3:])


def slice_position(ds, normal: np.ndarray | None) -> float | None:
    """Where this slice sits along the stack axis, in mm."""
    ipp = _get(ds, "ImagePositionPatient")
    if ipp is None or normal is None or len(ipp) != 3:
        return None
    return float(np.dot(np.asarray(ipp, dtype=float), normal))


def plane_of(normal: np.ndarray | None) -> str:
    """'axial' | 'sagittal' | 'coronal' | 'oblique' | 'unknown'.

    The dominant axis of the slice normal names the plane, but knee MRI is
    routinely prescribed oblique to the ligaments, so a normal that is not
    clearly dominated by one axis (< 0.75, i.e. more than ~41 deg off) is
    reported as oblique rather than silently rounded to the nearest plane.
    """
    if normal is None or not np.any(normal):
        return "unknown"
    n = np.abs(normal / np.linalg.norm(normal))
    axis = int(np.argmax(n))
    if n[axis] < 0.75:
        return "oblique"
    return ("sagittal", "coronal", "axial")[axis]


# Knee MRI protocols are named, not tagged: the weighting lives in
# SeriesDescription as free text. Order matters -- STIR and fat-sat qualifiers
# are checked before the plain weightings so "T2 FS" does not stop at "t2".
_WEIGHTING_RULES: list[tuple[str, str]] = [
    (r"\bstir\b", "STIR"),
    (r"\bt2\s*[\*x]|\bt2star|\bmerge\b|\bmedic\b", "T2*"),
    (r"\bpd\b|\bproton|\bdp\b", "PD"),
    (r"\bt1\b", "T1"),
    (r"\bt2\b", "T2"),
    (r"\bflair\b", "FLAIR"),
    (r"\bdwi\b|\bdiffus", "DWI"),
    (r"\b3d\b|\bmprage|\bvibe\b|\bfiesta|\btrufi|\bdess\b|\bfspgr", "3D"),
    (r"\bloc\b|\bscout|\blocali|\bsurvey", "localizer"),
]
_FATSAT = re.compile(r"\bfs\b|\bfat\s*sat|\bspair|\bspir\b|\bsat\b|\bstir\b|\bfatsat", re.I)


def infer_weighting(description: str | None, ds=None) -> str:
    """Best guess at sequence weighting from the series name, then the tags.

    A guess, and labelled as one: the series description is free text a
    technologist typed. Use it to filter candidate sequences, not as a feature
    you would stake a prediction on.
    """
    text = (description or "").lower()
    for pattern, name in _WEIGHTING_RULES:
        if re.search(pattern, text):
            return name
    if ds is not None:                      # fall back to acquisition timings
        tr, te = _num(_get(ds, "RepetitionTime")), _num(_get(ds, "EchoTime"))
        if tr is not None and te is not None:
            if tr < 900 and te < 30:
                return "T1"
            if tr >= 2000 and te >= 60:
                return "T2"
            if tr >= 2000 and te < 40:
                return "PD"
    return "unknown"


def sort_dicom_paths(datasets: Sequence, paths: Sequence[str]) -> list[int]:
    """Indices that put a series in through-plane anatomical order.

    Geometry first (position projected on the slice normal), InstanceNumber
    second, filename last. The geometric sort is the only one that is correct
    when a series was acquired interleaved or reconstructed out of order, but
    localizers and some derived series carry no position tags at all, hence
    the ladder. Sorting by filename alone silently shuffles slice order on any
    scanner whose exports are not zero-padded.
    """
    order = list(range(len(datasets)))
    normal = next((n for n in (slice_normal(d) for d in datasets) if n is not None), None)
    positions = [slice_position(d, normal) for d in datasets]
    if all(p is not None for p in positions) and len(set(positions)) > 1:
        return sorted(order, key=lambda i: positions[i])

    instance = [_num(_get(datasets[i], "InstanceNumber")) for i in order]
    if all(v is not None for v in instance) and len(set(instance)) > 1:
        return sorted(order, key=lambda i: instance[i])

    return sorted(order, key=lambda i: paths[i])


def scan_series(study_uid: str, series_uid: str, series_dir: str) -> dict:
    """Read every header in one series; return one flat metadata row.

    Never raises: a series that cannot be read comes back as a row with
    `scan_error` set, because a corpus-wide scan that dies on file 40,000 of
    50,000 has cost you the other 49,999.
    """
    row: dict[str, Any] = {
        "StudyInstanceUID": study_uid,
        "SeriesInstanceUID": series_uid,
        "series_dir": series_dir,
        "n_slices": 0,
        "missing_files": True,
        "scan_error": "",
        "paths": (),
    }

    paths = sorted(glob(os.path.join(series_dir, "*.dcm")))
    if not paths:
        row["scan_error"] = "no .dcm files"
        return row

    datasets, kept, failed = [], [], 0
    for p in paths:
        try:
            datasets.append(pydicom.dcmread(p, stop_before_pixels=True))
            kept.append(p)
        except Exception as exc:                        # unreadable single file
            failed += 1
            row["scan_error"] = f"{type(exc).__name__}: {exc}"
    if not datasets:
        return row

    try:
        order = sort_dicom_paths(datasets, kept)
    except Exception as exc:                            # bad geometry tags
        order = list(range(len(datasets)))
        row["scan_error"] = f"unsorted ({type(exc).__name__}: {exc})"

    datasets = [datasets[i] for i in order]
    ordered_paths = [kept[i] for i in order]
    head = datasets[0]

    normal = slice_normal(head)
    positions = [slice_position(d, normal) for d in datasets]
    known = [p for p in positions if p is not None]
    gaps = np.diff(known) if len(known) > 1 else np.array([])

    spacing = _get(head, "PixelSpacing", default=[None, None])
    rows_, cols_ = _num(_get(head, "Rows")), _num(_get(head, "Columns"))
    row_mm, col_mm = _num(spacing[0]), _num(spacing[1] if len(spacing) > 1 else None)
    desc = _get(head, "SeriesDescription", "ProtocolName", default="")

    row.update({
        "paths": tuple(ordered_paths),
        "n_slices": len(datasets),
        "n_unreadable": failed,
        "missing_files": False,

        # geometry
        "rows": int(rows_) if rows_ else None,
        "cols": int(cols_) if cols_ else None,
        "pixel_spacing_row": row_mm,
        "pixel_spacing_col": col_mm,
        "slice_thickness": _num(_get(head, "SliceThickness")),
        "spacing_between_slices": _num(_get(head, "SpacingBetweenSlices")),
        "slice_gap_median": float(np.median(np.abs(gaps))) if gaps.size else None,
        # A series whose slice spacing is not constant cannot be resampled as a
        # regular volume; flagging it beats discovering it in the model.
        "irregular_spacing": bool(gaps.size and np.ptp(np.abs(gaps)) > 0.51),
        "plane": plane_of(normal),
        "fov_row_mm": (rows_ * row_mm) if (rows_ and row_mm) else None,
        "fov_col_mm": (cols_ * col_mm) if (cols_ and col_mm) else None,
        "extent_mm": float(max(known) - min(known)) if len(known) > 1 else None,

        # protocol
        "modality": _get(head, "Modality", default=""),
        "series_description": str(desc),
        "series_number": _num(_get(head, "SeriesNumber")),
        "weighting": infer_weighting(str(desc), head),
        "fat_saturated": bool(_FATSAT.search(str(desc) or "")),
        "mr_acquisition_type": _get(head, "MRAcquisitionType", default=""),
        "scanning_sequence": str(_get(head, "ScanningSequence", default="")),
        "sequence_variant": str(_get(head, "SequenceVariant", default="")),
        "repetition_time": _num(_get(head, "RepetitionTime")),
        "echo_time": _num(_get(head, "EchoTime")),
        "inversion_time": _num(_get(head, "InversionTime")),
        "flip_angle": _num(_get(head, "FlipAngle")),
        "echo_train_length": _num(_get(head, "EchoTrainLength")),
        "magnetic_field_strength": _num(_get(head, "MagneticFieldStrength")),
        "body_part": str(_get(head, "BodyPartExamined", default="")),
        "laterality": str(_get(head, "ImageLaterality", "Laterality", default="")),

        # scanner
        "manufacturer": str(_get(head, "Manufacturer", default="")),
        "model": str(_get(head, "ManufacturerModelName", default="")),

        # patient
        "patient_id": str(_get(head, "PatientID", default="")),
        "patient_sex": str(_get(head, "PatientSex", default="")),
        "patient_age": _age_years(_get(head, "PatientAge")),

        # pixel pipeline
        "photometric": str(_get(head, "PhotometricInterpretation", default="")),
        "bits_stored": _num(_get(head, "BitsStored")),
        "rescale_slope": _num(_get(head, "RescaleSlope"), 1.0),
        "rescale_intercept": _num(_get(head, "RescaleIntercept"), 0.0),
        "window_center": _num(_get(head, "WindowCenter")),
        "window_width": _num(_get(head, "WindowWidth")),
    })
    return row


# =============================================================================
# 2. SeriesRecord -- one series, pixels on demand
# =============================================================================
@dataclass
class SeriesRecord:
    """One series: its ordered file paths, its metadata, its study's labels.

    Holds paths, not pixels. Constructing thousands of these costs nothing;
    `volume()` is where the IO happens.
    """

    study_uid: str
    series_uid: str
    paths: tuple[str, ...]
    meta: dict = field(default_factory=dict)
    labels: dict[str, float] = field(default_factory=dict)
    report: str = ""

    def __len__(self) -> int:
        return len(self.paths)

    @property
    def plane(self) -> str:
        return self.meta.get("plane", "unknown")

    @property
    def weighting(self) -> str:
        return self.meta.get("weighting", "unknown")

    def dicoms(self, metadata_only: bool = False) -> Iterator[pydicom.Dataset]:
        """Re-read the datasets, in anatomical order."""
        for p in self.paths:
            yield pydicom.dcmread(p, stop_before_pixels=metadata_only)

    def volume(self, dtype=np.float32, apply_rescale: bool = True,
               normalize: str | None = None) -> np.ndarray:
        """(n_slices, H, W) array, slices in through-plane order.

        normalize:
            None      raw stored values (after rescale)
            'minmax'  per-volume to [0, 1]
            'zscore'  per-volume zero mean, unit variance

        Per-volume, not per-slice, and per-volume, not per-dataset: MR
        intensities have no physical units and vary with coil, scanner and
        sequence, so a global normalisation constant is meaningless, while a
        per-slice one destroys the intensity relationship *between* slices
        that makes an effusion or a marrow oedema visible as it comes and goes
        through the stack.
        """
        if not self.paths:
            raise ValueError(f"series {self.series_uid} has no files")

        slices = []
        for p in self.paths:
            ds = pydicom.dcmread(p)
            arr = ds.pixel_array.astype(np.float32)
            if apply_rescale:
                arr = arr * _num(_get(ds, "RescaleSlope"), 1.0) + _num(_get(ds, "RescaleIntercept"), 0.0)
            if str(_get(ds, "PhotometricInterpretation", default="")) == "MONOCHROME1":
                arr = arr.max() - arr          # MONOCHROME1 stores inverted
            slices.append(arr)

        shapes = {s.shape for s in slices}
        if len(shapes) > 1:
            raise ValueError(
                f"series {self.series_uid} has ragged slices {sorted(shapes)}; "
                "resample or drop it before stacking"
            )

        vol = np.stack(slices).astype(dtype)
        if normalize == "minmax":
            lo, hi = float(vol.min()), float(vol.max())
            vol = (vol - lo) / (hi - lo) if hi > lo else np.zeros_like(vol)
        elif normalize == "zscore":
            sd = float(vol.std())
            vol = (vol - float(vol.mean())) / sd if sd > 0 else np.zeros_like(vol)
        elif normalize is not None:
            raise ValueError(f"unknown normalize={normalize!r}")
        return vol

    def label_vector(self, labels: Sequence[str]) -> np.ndarray:
        """Labels in a fixed column order -- the model's target vector."""
        return np.array([self.labels.get(k, np.nan) for k in labels], dtype=np.float32)


# =============================================================================
# 3. KneeDataset -- indexable, duck-types as a torch Dataset
# =============================================================================
class KneeDataset:
    """Indexable view over the records. Works as a `torch.utils.data.Dataset`.

    Deliberately not a torch subclass: torch is not imported here, so the same
    object works in a notebook, in a sklearn loop, or inside a DataLoader.
    """

    def __init__(self, records: Sequence[SeriesRecord], labels: Sequence[str],
                 transform: Callable[[dict], Any] | None = None,
                 load_pixels: bool = True, **volume_kwargs):
        self.records = list(records)
        self.labels = list(labels)
        self.transform = transform
        self.load_pixels = load_pixels
        self.volume_kwargs = volume_kwargs

    def __len__(self) -> int:
        return len(self.records)

    def __repr__(self) -> str:
        return f"KneeDataset({len(self)} series, {len(self.labels)} labels)"

    def __getitem__(self, idx: int) -> dict:
        r = self.records[idx]
        sample = {
            "volume": r.volume(**self.volume_kwargs) if self.load_pixels else None,
            "labels": r.label_vector(self.labels),
            "meta": r.meta,
            "study_uid": r.study_uid,
            "series_uid": r.series_uid,
        }
        return self.transform(sample) if self.transform else sample

    def filter(self, predicate: Callable[[SeriesRecord], bool]) -> "KneeDataset":
        """A new view over the records that pass. Cheap -- nothing is copied."""
        return KneeDataset([r for r in self.records if predicate(r)], self.labels,
                           self.transform, self.load_pixels, **self.volume_kwargs)


# =============================================================================
# 4. KneeDatasetBuilder -- the join
# =============================================================================
class KneeDatasetBuilder:
    """Reports -> labels, folders -> metadata, both -> one table.

        builder = KneeDatasetBuilder(DATA_PATH)
        df = builder.build()                    # scan + label + join
        builder.save("knee_index.parquet")      # cache; skip the scan next time
        ds = builder.dataset(normalize="zscore")

    The scan is the expensive step (one header read per slice), so it runs
    once, in a thread pool, and everything downstream works off the table.
    """

    # Candidate report columns, in the order they should be concatenated when a
    # dataset splits a report across several. Matched case-insensitively.
    REPORT_COLUMNS = (
        "report", "report_text", "radiology_report", "radiologist_report",
        "clinical_notes", "doctor_notes", "notes", "note", "text",
        "clinical_history", "history", "indication", "technique",
        "findings", "impression", "conclusion", "description",
    )

    def __init__(self, data_path: str, split: str = "train", *,
                 labeler: Any = None, extractor: Any = None,
                 report_col: str | Sequence[str] | None = None,
                 id_col: str = "StudyInstanceUID",
                 labels: Sequence[str] | None = None,
                 label_prefix: str = "", include_certainty: bool = True,
                 drop_empty_series: bool = True,
                 workers: int = 8, verbose: bool = True):
        """
        data_path         competition root (holds train.csv, train_series/, ...)
        split             'train' or 'test'; picks the csv names and image dir
        labeler           a ClinicalNoteLabeler; built with defaults if omitted
        extractor         a DICOMExtractor; built on data_path if omitted
        report_col        report column name(s). Auto-detected when None --
                          pass it explicitly once you know it.
        label_prefix      prefix for the label columns, e.g. 'y_'. Empty means
                          the label names themselves.
        include_certainty adds '<label>_certainty' columns (definite / probable
                          / negated / not_mentioned / ...). Keep them: they let
                          you weight a definite tear above a hedged one, or
                          drop hedged rows entirely, without re-running the
                          labeler.
        workers           threads for the header scan. IO-bound, so more than
                          cores is fine; drop to 1 to debug.
        """
        self.data_path = data_path
        self.split = split
        self.verbose = verbose
        self.workers = max(1, int(workers))
        self.drop_empty_series = drop_empty_series
        self.id_col = id_col
        self.label_prefix = label_prefix
        self.include_certainty = include_certainty

        self.study_csv = f"{split}.csv"
        self.series_csv = f"{split}_series.csv"
        self.series_dir = f"{split}_series"

        self.labeler = labeler if labeler is not None else ClinicalNoteLabeler()
        self.extractor = extractor if extractor is not None else DICOMExtractor(data_path)
        self.labels = list(labels if labels is not None else getattr(self.labeler, "labels", LABELS))
        self._report_col = report_col

        self.errors: list[dict] = []
        self._studies: pd.DataFrame | None = None
        self._series_meta: pd.DataFrame | None = None
        self._table: pd.DataFrame | None = None
        self._records: list[SeriesRecord] | None = None

    # -- small utilities --------------------------------------------------
    def _log(self, msg: str) -> None:
        if self.verbose:
            print(msg, flush=True)

    def _read_csv(self, name: str) -> pd.DataFrame:
        path = os.path.join(self.data_path, name)
        if not os.path.exists(path):
            raise FileNotFoundError(
                f"{path} not found. Set data_path/split to match your layout; "
                f"found: {sorted(os.listdir(self.data_path))[:20] if os.path.isdir(self.data_path) else 'no such dir'}"
            )
        return pd.read_csv(path)

    # -- the text side ----------------------------------------------------
    def resolve_report_columns(self, df: pd.DataFrame) -> list[str]:
        """Which column(s) hold the doctor's note.

        Auto-detection is a convenience for the first run, not a contract:
        competitions rename this column freely. It prefers a whole-report
        column, falls back to concatenating section columns (findings +
        impression), and raises with the actual column list rather than
        guessing wrong and labelling the entire corpus from nothing.
        """
        if self._report_col:
            cols = [self._report_col] if isinstance(self._report_col, str) else list(self._report_col)
            missing = [c for c in cols if c not in df.columns]
            if missing:
                raise KeyError(f"report column(s) {missing} not in {list(df.columns)}")
            return cols

        lower = {c.lower(): c for c in df.columns}
        whole = [lower[n] for n in self.REPORT_COLUMNS[:10] if n in lower]
        if whole:
            return whole[:1]
        sections = [lower[n] for n in self.REPORT_COLUMNS[10:] if n in lower]
        if sections:
            return sections

        # Last resort: the widest text column. Reports are long; ids are not.
        # `is_string_dtype` rather than `== object`: pandas >= 3 gives string
        # columns their own dtype and the object test silently finds nothing.
        text_cols = [c for c in df.columns
                     if c != self.id_col and (df[c].dtype == object
                                              or pd.api.types.is_string_dtype(df[c]))]
        if text_cols:
            widths = {c: df[c].astype(str).str.len().mean() for c in text_cols}
            best = max(widths, key=widths.get)
            if widths[best] > 40:
                warnings.warn(
                    f"No known report column; using {best!r} (mean length "
                    f"{widths[best]:.0f}). Pass report_col= to be explicit.")
                return [best]
        raise KeyError(
            f"No report column found in {list(df.columns)}. Pass report_col='<name>'.")

    def label_studies(self, df: pd.DataFrame | None = None,
                      relabel: bool = False) -> pd.DataFrame:
        """Run the labeler over every report. One row per study.

        Returns the original study csv plus the soft labels, their certainties
        and three diagnostics: `report_script`, `report_chars`, and
        `needs_review` (the labeler's own signal that it saw a script its
        vocabulary does not cover -- those rows are all-negative for the wrong
        reason and must not be trained on as negatives).
        """
        if self._studies is not None and df is None and not relabel:
            return self._studies                     # labelling is pure; do it once

        df = self._read_csv(self.study_csv) if df is None else df.copy()
        if self.id_col not in df.columns:
            raise KeyError(f"{self.id_col!r} not in {self.study_csv}: {list(df.columns)}")

        cols = self.resolve_report_columns(df)
        self._log(f"labelling {len(df)} reports from column(s) {cols}")
        reports = (df[cols].fillna("").astype(str)
                   .apply(lambda r: "\n".join(x for x in r if x.strip()), axis=1))
        df["report"] = reports

        rows = []
        for text in reports:
            res = self.labeler.extract(text)
            row: dict[str, Any] = {
                f"{self.label_prefix}{lab}": res[lab].value for lab in self.labels
            }
            if self.include_certainty:
                row.update({f"{lab}_certainty": res[lab].certainty for lab in self.labels})
            row["report_script"] = self.labeler.normalizer.detect_script(text)
            row["report_chars"] = len(text or "")
            row["needs_review"] = any(r.needs_review for r in res.values())
            rows.append(row)

        labelled = pd.concat([df.reset_index(drop=True), pd.DataFrame(rows)], axis=1)

        # A study csv that already ships ground-truth columns would be silently
        # overwritten by the labeler's estimates, which is how you end up
        # training on your own predictions. Keep both, renamed.
        dupes = [c for c in labelled.columns if list(labelled.columns).count(c) > 1]
        if dupes:
            warnings.warn(f"duplicate columns after labelling: {sorted(set(dupes))}")

        flagged = int(labelled["needs_review"].sum())
        if flagged:
            self._log(f"  {flagged} report(s) flagged needs_review "
                      f"(unsupported script) -- see builder.review_queue()")
        self._studies = labelled
        return labelled

    def review_queue(self, top: int = 20) -> pd.DataFrame:
        """The reports the labeler could not read, worst first."""
        df = self.study_table()
        bad = df[df["needs_review"]].copy()
        return bad[[self.id_col, "report_script", "report_chars", "report"]].head(top)

    # -- the pixel side ---------------------------------------------------
    def series_index(self) -> pd.DataFrame:
        """study -> series mapping, from the extractor."""
        mapping = self.extractor._getSeriesInstanceUID(self.series_csv)
        return pd.DataFrame(
            [(study, ser) for study, sers in mapping.items() for ser in sers],
            columns=["StudyInstanceUID", "SeriesInstanceUID"],
        )

    def scan(self, studies: Sequence[str] | None = None,
             limit_studies: int | None = None) -> pd.DataFrame:
        """Read every header and build the series metadata table.

        The expensive call. Threaded because it is IO-bound, and cached on the
        builder so `build()` is idempotent.
        """
        index = self.series_index()
        if studies is not None:
            index = index[index["StudyInstanceUID"].isin(set(studies))]
        if limit_studies:
            keep = index["StudyInstanceUID"].drop_duplicates().head(limit_studies)
            index = index[index["StudyInstanceUID"].isin(set(keep))]

        jobs = [(s, ser, os.path.join(self.data_path, self.series_dir, s, ser))
                for s, ser in zip(index["StudyInstanceUID"], index["SeriesInstanceUID"])]
        self._log(f"scanning {len(jobs)} series from "
                  f"{index['StudyInstanceUID'].nunique()} studies "
                  f"({self.workers} threads)")

        rows = []
        with ThreadPoolExecutor(max_workers=self.workers) as pool:
            for i, row in enumerate(pool.map(lambda j: scan_series(*j), jobs), 1):
                rows.append(row)
                if self.verbose and i % 250 == 0:
                    self._log(f"  {i}/{len(jobs)} series")

        meta = pd.DataFrame(rows)
        self.errors = [r for r in rows if r.get("scan_error") or r.get("missing_files")]
        if self.errors:
            self._log(f"  {len(self.errors)} series with problems -- see builder.errors")
        self._series_meta = meta
        return meta

    # -- the join ---------------------------------------------------------
    def build(self, limit_studies: int | None = None,
              rescan: bool = False) -> pd.DataFrame:
        """One row per series: series metadata + study metadata + labels.

        Left join from the series side, so a study whose folder is missing
        drops out with a warning instead of appearing as a label with no
        pixels behind it.
        """
        studies = self.label_studies()
        meta = (self.scan(studies=studies[self.id_col].tolist(), limit_studies=limit_studies)
                if (self._series_meta is None or rescan or limit_studies) else self._series_meta)

        if self.drop_empty_series:
            n = len(meta)
            meta = meta[meta["n_slices"] > 0].copy()
            if n - len(meta):
                self._log(f"  dropped {n - len(meta)} empty/unreadable series "
                          f"(drop_empty_series=False to keep them)")

        table = meta.merge(studies, on=self.id_col, how="left", suffixes=("", "_study"))

        orphans = int(table["report"].isna().sum()) if "report" in table else 0
        if orphans:
            warnings.warn(f"{orphans} series have no matching row in {self.study_csv}")

        table["n_series_in_study"] = table.groupby(self.id_col)["SeriesInstanceUID"].transform("count")
        # Group by this, never by row, when you split train/val: every series of
        # a study shares one report, so a row-level split leaks the label.
        table["group"] = table[self.id_col]

        front = [c for c in [self.id_col, "SeriesInstanceUID", "n_slices", "plane",
                             "weighting", "series_description", "n_series_in_study"]
                 if c in table.columns]
        rest = [c for c in table.columns if c not in front and c != "paths"]
        self._table = table[front + rest + (["paths"] if "paths" in table else [])]
        self._records = None
        self._log(f"built {len(self._table)} series x {len(self._table.columns)} columns")
        return self._table

    # -- accessors --------------------------------------------------------
    def study_table(self) -> pd.DataFrame:
        """Study-level view: one row per study, labels plus series counts."""
        if self._table is None:
            return self.label_studies()
        agg = (self._table.groupby(self.id_col)
               .agg(n_series=("SeriesInstanceUID", "count"),
                    n_slices=("n_slices", "sum"),
                    planes=("plane", lambda s: sorted(set(s))),
                    weightings=("weighting", lambda s: sorted(set(s))))
               .reset_index())
        studies = self.label_studies()
        return studies.merge(agg, on=self.id_col, how="left")

    @property
    def table(self) -> pd.DataFrame:
        if self._table is None:
            self.build()
        return self._table

    @property
    def records(self) -> list[SeriesRecord]:
        """SeriesRecord per row, labels attached, pixels still on disk."""
        if self._records is None:
            df = self.table
            label_cols = [f"{self.label_prefix}{l}" for l in self.labels]
            meta_cols = [c for c in df.columns if c not in label_cols + ["paths"]]
            self._records = [
                SeriesRecord(
                    study_uid=r[self.id_col],
                    series_uid=r["SeriesInstanceUID"],
                    paths=tuple(r.get("paths") or ()),
                    meta={c: r[c] for c in meta_cols},
                    labels={l: r.get(f"{self.label_prefix}{l}") for l in self.labels},
                    report=r.get("report", "") or "",
                )
                for r in df.to_dict("records")
            ]
        return self._records

    def dataset(self, transform=None, load_pixels: bool = True, **volume_kwargs) -> KneeDataset:
        """Indexable dataset over the records. Extra kwargs go to `volume()`."""
        return KneeDataset(self.records, self.labels, transform, load_pixels, **volume_kwargs)

    # -- cache ------------------------------------------------------------
    def save(self, path: str) -> str:
        """Persist the built table so the next session skips the scan.

        `paths` is a tuple per row, which parquet handles and csv does not, so
        it is JSON-encoded on the way out and decoded on the way back in.
        """
        df = self.table.copy()
        df["paths"] = df["paths"].apply(lambda p: json.dumps(list(p or ())))
        if path.endswith(".parquet"):
            try:
                df.to_parquet(path, index=False)
            except ImportError:                      # no pyarrow/fastparquet here
                path = path[: -len(".parquet")] + ".csv"
                warnings.warn(f"no parquet engine installed; wrote {path} instead")
                df.to_csv(path, index=False)
        else:
            df.to_csv(path, index=False)
        self._log(f"wrote {len(df)} rows to {path}")
        return path

    @classmethod
    def load(cls, path: str, data_path: str = "", **kwargs) -> "KneeDatasetBuilder":
        """Rebuild a builder from a cached table. No scan, no labelling."""
        df = pd.read_parquet(path) if path.endswith(".parquet") else pd.read_csv(path)
        df["paths"] = df["paths"].apply(
            lambda s: tuple(json.loads(s)) if isinstance(s, str) else tuple(s or ()))
        obj = cls(data_path or ".", verbose=False, **kwargs)
        obj._table = df
        return obj

    def __repr__(self) -> str:
        n = "unbuilt" if self._table is None else f"{len(self._table)} series"
        return f"KneeDatasetBuilder({self.data_path!r}, split={self.split!r}, {n})"


def build_knee_dataset(data_path: str, **kwargs) -> tuple[pd.DataFrame, KneeDataset]:
    """One-liner for the common case: the table and the dataset.

        df, ds = build_knee_dataset(DATA_PATH)
    """
    builder = KneeDatasetBuilder(data_path, **kwargs)
    return builder.build(), builder.dataset()

In [ ]:
# ============================================================================
# Quick start
# ============================================================================
builder = KneeDatasetBuilder(DATA_PATH, workers=8)

df = builder.build()          # scan headers, label reports, join
print(df.shape)
df.head()


In [ ]:
# ---------------------------------------------------------------------------
# What went wrong, and what to do next
# ---------------------------------------------------------------------------
# Check these two before trusting a single row.
print(len(builder.errors), "series could not be scanned")     # missing / unreadable
display(builder.review_queue())                               # reports in an
                                                              # unsupported script:
                                                              # all-negative for the
                                                              # wrong reason -- exclude
                                                              # them, do not train on
                                                              # them as negatives.
print(builder.labeler.coverage_report(zip(df["StudyInstanceUID"], df["report"])))

# The header scan is the slow part and only changes when the data does.
builder.save("knee_index.parquet")
# next session:  builder = KneeDatasetBuilder.load("knee_index.parquet")

# Sagittal fat-suppressed series -- the ones the menisci are actually read on.
ds = (builder.dataset(normalize="zscore")
             .filter(lambda r: r.plane == "sagittal" and r.meta["fat_saturated"]))
print(ds)

sample = ds[0]
print(sample["volume"].shape, sample["labels"].round(2))

# Split on `group` (the study), never on the row: every series of a study shares
# one report, so a row-level split puts the same label on both sides.
#   from sklearn.model_selection import GroupShuffleSplit
#   tr, va = next(GroupShuffleSplit(test_size=0.2, random_state=0)
#                 .split(df, groups=df["group"]))
#
# `ds` already duck-types as a torch Dataset:
#   from torch.utils.data import DataLoader
#   loader = DataLoader(ds, batch_size=1, collate_fn=lambda b: b)   # ragged depths
